# Lab 8 Student Version (Google Colab)
## Text Classification with LSTM (IMDb Sentiment)

This notebook is a **student-learning version** of Lab 8. It includes **every topic and step** from the lab guide and adds short explanations of **what you are learning** and **why each step matters**.

## Lab overview
In this lab, you will build a sentiment classification system using a **Long Short-Term Memory (LSTM)** network in PyTorch. You will use a pre-downloaded IMDb dataset in CSV format and complete four major tasks:
1) preprocessing raw text data  
2) creating a custom PyTorch dataset + collate function + DataLoaders  
3) building and training an LSTM model  
4) evaluating with classification metrics, plotting performance, and writing an inference function for new reviews

### In this lab, you will
- Load and preprocess the IMDb review dataset
- Tokenize and vectorize reviews using **spaCy** and a **custom vocabulary**
- Build an LSTM-based sentiment classifier in PyTorch
- Evaluate using accuracy, precision, recall, F1-score, and a confusion matrix
- Visualize training and validation performance over time
- Make predictions using a user-defined text inference function

### Estimated completion time
- **30 minutes**

### Runtime type for this lab (important)
- Set Runtime type to: **T4 GPU**


# Task 1: Loading and preprocessing the IMDb dataset

### What you are learning
You are learning how to prepare raw text for deep learning:
- install NLP tools (**spaCy**) and its English model
- upload and load a CSV dataset
- sample a smaller subset for faster training
- map labels into a format suitable for neural networks
- tokenize text and build a vocabulary (top 10,000 words + special tokens)

### Step 1: Install dependencies (spaCy + English model)
Run the cell below **once** to install dependencies.

> The lab guide also instructs you to switch runtime to **T4 GPU** and then restart the session:
> - **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**
> - **Runtime → Restart session**
>
> After restarting, re-run the notebook cells from the top.


In [ ]:
!pip install spacy --quiet
!python -m spacy download en_core_web_sm


### Step 2: Import libraries


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import spacy
import random
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


### Step 3: Upload the dataset CSV

Upload **IMDB Dataset.csv** when prompted.


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


### Step 4: Load dataset, sample 6000, map labels, split train/val/test


In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df = df.sample(n=6000, random_state=42).reset_index(drop=True)

label_map = {"positive": 1, "negative": 0}
df["label"] = df["sentiment"].map(label_map)

train_df = df[:4800]
valid_df = df[4800:5400]
test_df  = df[5400:]

print("Total rows:", len(df))
print("Train/Valid/Test:", len(train_df), len(valid_df), len(test_df))


### Step 5: Tokenize with spaCy and build a vocabulary


In [ ]:
nlp = spacy.load("en_core_web_sm")

def tokenize(text):
    return [token.text.lower() for token in nlp(text) if token.is_alpha]

counter = Counter()
for review in train_df["review"]:
    counter.update(tokenize(review))

vocab = {"<pad>": 0, "<unk>": 1}
for idx, (word, _) in enumerate(counter.most_common(10000), start=2):
    vocab[word] = idx

print("Vocab size:", len(vocab))


# Task 2: Creating Dataset, collate function, and DataLoaders


In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["review"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = tokenize(self.texts[idx])
        indices = [vocab.get(token, vocab["<unk>"]) for token in tokens]
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)


In [ ]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in texts])
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.stack(labels)
    return padded_texts, lengths, labels


In [ ]:
BATCH_SIZE = 64

train_ds = IMDBDataset(train_df)
valid_ds = IMDBDataset(valid_df)
test_ds  = IMDBDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

x, lengths, y = next(iter(train_loader))
print("padded_texts:", x.shape, "| lengths:", lengths.shape, "| labels:", y.shape)


# Task 3: Building, training, and visualizing LSTM model


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.5)

    def forward(self, text, lengths):
        embedded = self.embedding(text)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        hidden = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        return self.fc(self.dropout(hidden)).squeeze(1)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMClassifier(len(vocab), 100, 128, 1).to(device)

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss().to(device)

def binary_accuracy(preds, y):
    rounded = torch.round(torch.sigmoid(preds))
    return (rounded == y).float().mean()

def train_epoch(model, loader):
    model.train()
    total_loss, total_acc = 0.0, 0.0
    for x, lengths, y in loader:
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        optimizer.zero_grad()
        preds = model(x, lengths)
        loss = criterion(preds, y)
        acc = binary_accuracy(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += acc.item()
    return total_loss / len(loader), total_acc / len(loader)

def evaluate(model, loader):
    model.eval()
    total_loss, total_acc = 0.0, 0.0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            preds = model(x, lengths)
            loss = criterion(preds, y)
            acc = binary_accuracy(preds, y)
            total_loss += loss.item()
            total_acc += acc.item()
    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(3):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, valid_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.3f}, Acc={train_acc:.2f} | Val Loss={val_loss:.3f}, Acc={val_acc:.2f}")


In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.title("Accuracy Curve")
plt.legend()
plt.show()


# Task 4: Evaluating and using the model for predictions


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, lengths, y in test_loader:
        x, lengths = x.to(device), lengths.to(device)
        preds = torch.round(torch.sigmoid(model(x, lengths)))
        all_preds.extend(preds.cpu().numpy().reshape(-1).tolist())
        all_labels.extend(y.numpy().reshape(-1).tolist())

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec  = recall_score(all_labels, all_preds)
f1   = f1_score(all_labels, all_preds)

print(f"Accuracy: {acc:.2f}, Precision: {prec:.2f}, Recall: {rec:.2f}, F1: {f1:.2f}")


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()


In [ ]:
def predict_sentiment(text):
    model.eval()
    tokens = tokenize(text)
    indices = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
    length = torch.tensor([len(indices)]).to(device)
    with torch.no_grad():
        output = torch.sigmoid(model(tensor, length))
    return "Positive 🙂" if output.item() >= 0.5 else "Negative 🙁"


In [ ]:
# Example prediction
predict_sentiment("This movie is absolutely fantastic and heartwarming!")


# Lab review

1. Which metric is best for evaluating imbalanced binary classification?

A. Accuracy  
B. Recall  
C. F1-Score  
D. Confusion matrix  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answer (click to expand)</strong></summary>

**C. F1-Score**

</details>
